# 08. Post-specification robustness and uncertainty

This reporting notebook is read-only with respect to the frozen main policy artifact. It loads outputs written by standalone robustness scripts. Behavioral-parameter ranges condition on selected calendars; they exclude calendar-selection, PPML-estimation, planning-origin, and historical-state uncertainty.

In [1]:
from pathlib import Path
import pandas as pd
import sys

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run from the repository or notebooks directory.')
MAIN_ARTIFACT = PROJECT_ROOT / 'artifacts' / 'policy' / 'policy_optimization.pkl'
ROBUSTNESS_DIR = PROJECT_ROOT / 'results' / 'robustness'
assert MAIN_ARTIFACT.exists(), 'Frozen main policy artifact is missing.'
ROBUSTNESS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

## Reproduce the robustness outputs

Run the next two cells in order. They execute the robustness computations in this notebook kernel and retain the resulting DataFrames for inspection. They never overwrite `artifacts/policy/policy_optimization.pkl`. The first takes roughly 6 minutes; the second is the expensive alternative-PPML reoptimization and took roughly 11 minutes in the final run.

In [ ]:
# A and C: execute in-kernel. No PPML fitting and no policy reoptimization occur here.
from run_final_robustness_reporting import run_reporting_robustness

reporting_results = run_reporting_robustness()
same_horizon_grid = reporting_results['same_horizon_grid']
normalization_peaks = reporting_results['same_horizon_peaks']
parent_level_contrasts = reporting_results['fixed_calendar_parent_contrasts']
uncertainty_peaks = reporting_results['fixed_calendar_summary']
print(f"Completed A and C in {reporting_results['runtime_seconds']:.1f} seconds.")

In [ ]:
# B: execute the expensive PPML fit, common-origin profile, and full reoptimization in-kernel.
# It reuses final behavioral draws/actions and writes only under robustness paths.
from run_alternative_ppml_robustness import run_alternative_ppml_robustness

alternative_results = run_alternative_ppml_robustness()
alternative_prediction_grid = alternative_results['predictions']
alternative_weekly_profile = alternative_results['weekly_profile_table']
alternative_policy_grid = alternative_results['full_grid']
alternative_summary = alternative_results['summary']
alternative_vs_main = alternative_results['comparison']
display(alternative_summary)
print(f"Completed B in {alternative_results['runtime_seconds']:.1f} seconds.")

## A. Same-horizon normalization

The 12-week myopic-planning-profit percentage remains the operational primary measure. The 48-week percentage is reported only as a same-horizon normalization.

In [2]:
display(normalization_peaks)

,reimbursement_share,capacity,myopic_planning_profit_12w,myopic_profit_48w,delta_plan,delta_plan_pct_myopic_planning_12w,delta_plan_pct_myopic_same_horizon_48w,delta_disp,delta_disp_pct_myopic_planning_12w,delta_disp_pct_myopic_same_horizon_48w,delta_total,delta_total_pct_myopic_planning_12w,delta_total_pct_myopic_same_horizon_48w
0,0.90,1,78175.457261,291383.575426,28.951565,0.037034,0.009936,1006.686472,1.287727,0.345485,1035.638037,1.324761,0.355421
1,0.89,2,77339.663875,290590.961172,96.069320,0.124217,0.033060,1471.287825,1.902372,0.506309,1567.357146,2.026589,0.539369
2,0.90,3,77559.471114,290733.039281,135.105142,0.174196,0.046471,1629.231746,2.100623,0.560388,1764.336889,2.274818,0.606858
3,0.90,8,77377.688769,290691.007058,126.536578,0.163531,0.043530,1682.744957,2.174716,0.578878,1809.281534,2.338247,0.622407


## B. Alternative demand-model robustness

This uses the simpler `product_promotion` PPML profile but retains the frozen behavioral draws, actions, economics, initialization, horizon, and optimizer.

In [3]:
display(alternative_summary)

,specification,capacity,peak_delta_total,peak_lambda,peak_delta_plan,peak_delta_disp,displacement_share_at_peak,piM_transitions,piN_transitions,piD_transitions,piM_transition_locations,piN_transition_locations,piD_transition_locations
0,alternative_product_promotion,1,1225.294585,0.89,293.543876,931.750708,0.760430,10,13,13,0.75|0.78|0.79|0.81|0.83|0.85|0.87|0.89|0.93|0.95,0.75|0.78|0.79|0.81|0.83|0.85|0.87|0.88|0.89|0...,0.79|0.81|0.82|0.84|0.85|0.86|0.88|0.89|0.90|0...
1,alternative_product_promotion,2,1777.642361,0.89,185.017185,1592.625176,0.895920,15,15,14,0.75|0.78|0.79|0.81|0.83|0.85|0.86|0.87|0.89|0...,0.75|0.78|0.79|0.81|0.83|0.85|0.86|0.87|0.89|0...,0.79|0.81|0.82|0.84|0.85|0.88|0.89|0.94|0.95|0...
2,alternative_product_promotion,3,2023.406975,0.90,158.035050,1865.371925,0.921897,12,14,17,0.75|0.78|0.79|0.83|0.85|0.86|0.87|0.89|0.90|0...,0.75|0.78|0.79|0.83|0.85|0.86|0.87|0.89|0.90|0...,0.79|0.81|0.82|0.84|0.85|0.86|0.88|0.89|0.91|0...
3,alternative_product_promotion,8,2072.048418,0.90,138.147383,1933.901035,0.933328,10,10,14,0.75|0.78|0.79|0.83|0.85|0.86|0.87|0.89|0.90|0.93,0.75|0.78|0.79|0.83|0.85|0.86|0.87|0.89|0.90|0.93,0.79|0.81|0.82|0.84|0.85|0.86|0.88|0.89|0.93|0...


## C. Behavioral-parameter uncertainty conditional on selected calendars

In [4]:
display(uncertainty_peaks)

,capacity,reimbursement_share,component,p05,p50,p95,label,bootstrap_parents
0,1,0.90,delta_plan,-12.471143,28.692373,73.656340,behavioral-parameter uncertainty conditional o...,1000
1,1,0.90,delta_disp,367.116632,993.894388,1760.077838,behavioral-parameter uncertainty conditional o...,1000
2,1,0.90,delta_total,384.495739,1029.116134,1789.029203,behavioral-parameter uncertainty conditional o...,1000
3,2,0.89,delta_plan,-6.110946,94.533352,199.958319,behavioral-parameter uncertainty conditional o...,1000
4,2,0.89,delta_disp,804.301288,1457.913529,2222.911648,behavioral-parameter uncertainty conditional o...,1000
5,2,0.89,delta_total,844.772411,1563.714408,2367.357552,behavioral-parameter uncertainty conditional o...,1000
6,3,0.90,delta_plan,78.344759,134.788036,193.625072,behavioral-parameter uncertainty conditional o...,1000
7,3,0.90,delta_disp,570.343729,1495.824395,2856.760473,behavioral-parameter uncertainty conditional o...,1000
8,3,0.90,delta_total,709.340174,1653.711445,2967.510067,behavioral-parameter uncertainty conditional o...,1000
9,8,0.90,delta_plan,72.910537,125.539625,182.606955,behavioral-parameter uncertainty conditional o...,1000
